## Statistical data analysis. Exam 

**(!) <span style="color:blue"> Add your name to the file name instead of [Name]</span>**</br>
Example: Stat_analysis_23-24_exam_Ivanov Ivan.ipynb

Submit your notebook to network folder of your class (R610 / R611 / R612) **Submit_Stat analysis_[class name]**

Unlocked web sites to visit:</br>
* google search
* https://docs.scipy.org/doc/scipy/tutorial/index.html
* https://docs.python.org/3/library/
* https://numpy.org/
* https://pandas.pydata.org/docs/user_guide/index.html
* https://matplotlib.org/
* https://stackoverflow.com/

In [ ]:
import numpy as np
import random
import plotly.graph_objects as go

**Question 1 [12 pts]**. 

Consider the following random process $S_k^{\alpha},\ k\ge 0,\ S_0^{\alpha}=0$ which is called **Skew Random Walk**: </br>
transition probabilities $p_{0,1}=\alpha,\ p_{0, -1}=1-\alpha$ and $p_{i,i-1}=p_{i,i+1}=0.5,\, i\ne 0$.</br>
Simply speaking, if $S^{\alpha}_k\ne 0$ then next step is $\pm 1$ (i.e. $S^{\alpha}_{k+1}=S^{\alpha}_{k}\pm 1$) with probability $0.5$ like simple symmetric Random walk. But if $S^{\alpha}_k = 0$ then next step is $-1$ with probability $1-\alpha$ (i.e. $S^{\alpha}_{k+1}=S^{\alpha}_{k}-1$) and $1$ with probability $\alpha$ (i.e. $S^{\alpha}_{k+1}=S^{\alpha}_{k}+1$).

Let $\alpha=0.85$.</br>
1) Output 10 paths of $S^{\alpha}_k,\ k=0,\ldots, 500$. Output these paths in form of scatterplots (mode=lines). </br>

In [ ]:
alpha = 0.85
n = 500

def path(n):
    seq = [0]
    for i in range(0,n+1):
        if seq[i] == 0:
            step = random.choices([-1, 1], weights = [1-alpha, alpha])[0]
        else:
            step = random.choices([-1, 1], weights = [0.5, 0.5])[0]
        seq.append(seq[i]+step)
    return seq

In [ ]:
N=10

t = np.linspace(0, n, n+1)

plot = go.Figure()

for i in range(0,N):
    graph = go.Scatter(x = t, y = path(n))
    plot.add_trace(graph)
    
plot.show()

2) Output 95\% confidence interval for the last value $S^{\alpha}_{100}$ based on 10 000 paths.

In [ ]:
N = 10000
n = 100
last = []
for i in range(0,N):
    last.append(path(n)[-1])
np.quantile(last, q = [0.025, 0.975])

**Question 2 [12 pts]**

In [ ]:
from scipy.stats import rv_discrete
from scipy.stats import binom

Let $X$ be a random variable with the following distribution:
$$
P(X=k) = \frac{1}{5}\cdot C_{100}^k\cdot (0.25)^k\cdot (0.75)^{100-k}+\frac{4}{5}\cdot C_{100}^k\cdot (0.75)^k\cdot (0.25)^{100-k},\ k=0,\ldots, 100
$$

1) Derive a class from rv_discrete (call it *mix_binom*) which represents distribution above.</br>
Hint: you may find useful to use binom.pmf function when coding probabilities $C_{100}^k\cdot p^k\cdot (1-p)^{100-k}$
2) Create an instance of this class and generate a sample $X_1,\ldots, X_n$ of size $n=2500$.

In [ ]:
class mix_binom(rv_discrete):
    def _pmf(self, k):
        return 0.2*binom.pmf(k, 100, 0.25)+0.8*binom.pmf(k,100, 0.75)

In [ ]:
distr = mix_binom()

In [ ]:
distr.mean()

In [ ]:
sample = distr.rvs(size = 2500)

3) Output a histogram for sample produced

In [ ]:
plot = go.Figure()

hist = go.Histogram(x = sample)
plot.add_trace(hist)

4) Output difference $|\overline X - \mathsf E(X)|,$ where $\mathsf E(X)$ is computed using *mix_binom* class functionality. 

In [ ]:
abs(np.mean(sample)-distr.mean())

**Question 3 [16 pts + 8 bonus pts]**

In [ ]:
def bm_simulations(n_paths, granularity, max_time):
    n_steps = granularity*max_time

    paths = []
    for _ in range(0, n_paths):
        seq = np.array(random.choices([-1, 1], weights = [0.5, 0.5], k = n_steps))
        path = np.cumsum(seq)/np.sqrt(granularity)
        # insert 0 to the beginning as an entry point
        path = np.insert(path,0,0)
        paths.append(path)    
   
    return paths

Consider financial instrument with payoff $f(S_T)=S_T^2-K$ under Black-Scholes model. Class for this instrument is in place:

In [ ]:
class instrument:
    def __init__ (self, K, T):
        self.strike = K
        self.maturity = T
        
    def payoff(self, S):
        return S**2-self.strike

Let $S_0=10,\, K=90$. Black-Scholes model parameters $r=10\%$, volatility $\sigma=1$. Your task is to price this financial instrument with half a year maturity.</br>

In [ ]:
S0 = 10
K = 90
T = 0.5
r = 0.1
sigma = 1

 You need to write only *pricer* class (type of pricer - Monte-Carlo or Analytic is of your choice) with price(instrument) function. Full points are awarder for any correctly written pricer. No need to write *model* class, plug model parameters to price() function as they are.</br>
If you implement both pricers then +8 bonus points are awarded. In this case analytic_pricer and mc_pricer should have parent pricers.</br>
**Output**: PV of an instrument.

In [ ]:
instr = instrument(K, T)

In [ ]:
class pricer:
    def __init__(self, mod):
        self.model = mod
        
    def price(self, instr):
        pass

In [ ]:
class analytic_pricer(pricer):
    
    def __init__(self):
        pass
        
    def price(self, instr):
        K = instr.strike
        ttm = instr.maturity
        
        return S0**2*np.exp((r+sigma**2)*ttm)-K*np.exp(-r*ttm)

In [ ]:
analytic_pricer().price(instr)

In [ ]:
from scipy.stats import norm

In [ ]:
class monte_carlo_pricer(pricer):
    def __init__(self, N_paths):
        self.N = N_paths        
        
    def price(self, instr):
        ttm = instr.maturity
        norm_rvs = norm.rvs(size = self.N)
        
        S_T = []
        for z in norm_rvs:
            degree = sigma * np.sqrt(ttm) * z + (r - sigma**2/2) * ttm
            S_T.append(S0 * np.exp(degree))
        
        payoffs = [instr.payoff(sti) for sti in S_T]  
        discounted_payoffs = np.array(payoffs) * np.exp(-r*ttm)
    
        return np.mean(discounted_payoffs)   

In [ ]:
n_paths = 100000
mc_pricer = monte_carlo_pricer(n_paths)
mc_price_ = mc_pricer.price(instr)
mc_price_

**Question 4 [10 pts]**

Consider differential equation and finite difference scheme from Problem 4.</br>
Implement finite difference scheme. Output 4 scatterplots (mode=lines) on the same plot: exact solution $y=y(x)$ and 3 solutions $(y_0,\ldots, y_n)$ of finite difference schemes for $n=10,\,20$ and $50$.

In [ ]:
initial = 2
Ns = [10, 20, 50]

solutions = []
for N in Ns:
    x_ = np.linspace(0,1,N+1)
    h = 1/N
    yh = [initial]
    for i in range(0, N):
        yh.append(yh[i]*(1-2*(i+1)*h**2))
    solutions.append([x_, yh])

In [ ]:
plot = go.Figure()

x_ = np.linspace(0,1,1001)
y_ = initial*np.exp(-x_**2)
graph = go.Scatter(x = x_, y = y_, name = 'exact')
plot.add_trace(graph)

for i in range(len(Ns)):
    graph1 = go.Scatter(x = solutions[i][0], y = solutions[i][1], mode ='lines', name = 'N='+str(Ns[i]))
    plot.add_trace(graph1)

plot.show()